In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the two coefficients of $y[n] = b_0\,x[n] + b_1\,x[n-1]$. On the left,
the purple stems are the output for the chapter's square wave (faint blue).
On the right, the red curve is the filter's gain at every frequency, against
the flat gray line of the white-noise input. The audio card plays white noise
through the filter. Try $b_0 = b_1$ for a low-pass, and $b_0 = -b_1$ for a
high-pass.

In [ ]:
# hide
# autorun
B0, B1 = 1.0, 1.0                   # start at the chapter's first filter, y1

PURPLE = "#6E3B87"                  # the chapter's output purple
SR = 44100

# the chapter's square wave: a ten-sample period, zero before n = 0
N_SHOW = 32
NS = np.arange(N_SHOW)
XSQ = np.where((NS // 5) % 2 == 0, 1.0, -1.0)
XSQ_PREV = np.concatenate([[0.0], XSQ[:-1]])

F = np.linspace(0.0, SR / 2, 600)
EJW = np.exp(-2j * np.pi * F / SR)     # e^{-j 2 pi f / f_s}: one sample of delay

# quiet white noise, so even a gain of 2 keeps the peak at -6 dBFS
NOISE = 0.25 * np.random.default_rng(0).uniform(-1.0, 1.0, 2 * SR)
NOISE_PREV = np.concatenate([[0.0], NOISE[:-1]])


def stems(n, v):
    """x and y arrays that draw stems from 0 to v at each n, as one trace."""
    xs = np.repeat(n.astype(float), 3)
    ys = np.column_stack([np.zeros(len(v)), v, np.full(len(v), np.nan)]).ravel()
    xs[2::3] = np.nan
    return xs, ys


def output(b0, b1, XSQ=XSQ, XSQ_PREV=XSQ_PREV):
    return b0 * XSQ + b1 * XSQ_PREV


def response(b0, b1, EJW=EJW):
    return np.abs(b0 + b1 * EJW)


def figure():
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12)
    xs, ys = stems(NS, XSQ)
    fig.add_scatter(x=xs, y=ys, mode="lines", opacity=0.35,
                    line=dict(color=BLUE, width=1.4), row=1, col=1)
    fig.add_scatter(x=NS, y=XSQ, mode="markers", opacity=0.35,
                    marker=dict(color=BLUE, size=6), row=1, col=1)
    y0 = output(B0, B1)
    xs, ys = stems(NS, y0)
    fig.add_scatter(x=xs, y=ys, mode="lines",
                    line=dict(color=PURPLE, width=1.6), row=1, col=1)
    fig.add_scatter(x=NS, y=y0, mode="markers",
                    marker=dict(color=PURPLE, size=7), row=1, col=1)
    fig.add_scatter(x=[0, SR / 2000], y=[1, 1], mode="lines",
                    line=dict(color=IRON, width=1.4, dash="dash"), row=1, col=2)
    fig.add_scatter(x=F / 1000, y=response(B0, B1), mode="lines",
                    line=dict(color=RED, width=2.2), row=1, col=2)
    fig.update_xaxes(range=[-0.8, N_SHOW - 0.2], title_text="Sample index n",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-2.3, 2.3], title_text="Amplitude",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, SR / 2000], title_text="Frequency (kHz)",
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[0, 2.1], title_text="Gain",
                     fixedrange=True, row=1, col=2)
    return fig


def controls(fig):
    b0 = widgets.FloatSlider(description="$b_0$", min=-1.0, max=1.0, step=0.05,
                             value=B0)
    b1 = widgets.FloatSlider(description="$b_1$", min=-1.0, max=1.0, step=0.05,
                             value=B1)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(b0, b1, NS=NS, stems=stems, output=output, response=response,
               readout=readout):
        y = output(b0, b1)
        xs, ys = stems(NS, y)
        with fig.batch_update():
            fig.data[2].y = ys
            fig.data[3].y = y
            fig.data[5].y = response(b0, b1)
        dc, nyq = abs(b0 + b1), abs(b0 - b1)
        if dc > nyq + 0.05:
            kind = "passes lows more than highs: a low-pass"
        elif nyq > dc + 0.05:
            kind = "passes highs more than lows: a high-pass"
        else:
            kind = "the same gain at both ends"
        readout.value = (f"<span style='font-size:0.9em'>gain at DC "
                         f"|<i>b</i><sub>0</sub> + <i>b</i><sub>1</sub>| = {dc:.2f}"
                         f" &nbsp;·&nbsp; at Nyquist "
                         f"|<i>b</i><sub>0</sub> &minus; <i>b</i><sub>1</sub>| = "
                         f"{nyq:.2f} &nbsp;·&nbsp; {kind}</span>")

    widgets.interactive_output(update, {"b0": b0, "b1": b1})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(NOISE=NOISE, NOISE_PREV=NOISE_PREV):
        y = b0.value * NOISE + b1.value * NOISE_PREV
        y[:441] *= np.linspace(0, 1, 441)
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (b0, b1):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([b0, b1, readout, out, gate])

icm_plotly.show(figure, controls)